## 0. 환경 설정

In [ ]:
#!/usr/bin/env python3
"""
DuckDB에서 직접 n 명 Patient 코호트 생성 (수정본)
- 감염 관련 입원 전체 추출
- 대표 입원
- MDRO 코드 수정
"""

import duckdb
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ============================================
# 경로 설정
# ============================================

DB_PATH = '../mimic_total.duckdb'
OUTPUT_CSV = '../outputs/cohort_patients_v4.csv'
OUTPUT_JSON = '../outputs/cohort_patients_v4.json'

print("=" * 70)
print("n 명 Patient 코호트 생성 (수정본 v4)")
print("=" * 70)

DuckDB에서 10명 Patient 코호트 생성 (수정본 v4)


### 1) DuckDB 연결

In [21]:
# ============================================
# Step 0: DuckDB 연결
# ============================================

print("\n[Step 0] DuckDB 연결...")
print(f"  경로: {DB_PATH}")

conn = duckdb.connect(DB_PATH, read_only=True)
print("  ✅ 연결 성공")


[Step 0] DuckDB 연결...
  경로: ../mimic_total.duckdb
  ✅ 연결 성공


### 2) 일반 병동 정의

In [15]:
conn.execute("""
    SELECT DISTINCT careunit, COUNT(*) as cnt 
    FROM transfers 
    WHERE careunit IS NOT NULL
    GROUP BY careunit 
    ORDER BY cnt DESC
""").df()

,careunit,cnt
0,Emergency Department,784400
1,UNKNOWN,546196
2,Medicine,191862
3,Emergency Department Observation,101347
4,Discharge Lounge,71241
5,Med/Surg,56907
6,Medicine/Cardiology,53897
7,Neurology,49749
8,Transplant,37907
9,Hematology/Oncology,37354


---

## 1. MDRO 환자 추출

### 1) MDRO 테이블 생성 (MRSA, CRE만 - 3계열 동반 내성)

In [22]:
print("\n[Step 1] microbiologyevents에서 MDRO 추출 (MRSA, CRE - 3계열 동반)...")

mdro_query = """
CREATE OR REPLACE TEMP TABLE mdro_admissions AS

WITH mrsa_cases AS (
    -- MRSA: 베타락탐계(필수) + 마크로라이드계 + 퀴놀론계 (3계열 모두 내성)
    SELECT hadm_id, 'MRSA' AS mdro_type
    FROM (
        SELECT 
            hadm_id,
            -- 베타락탐계 (필수): 옥사실린, 세팔로스포린
            MAX(CASE WHEN ab_name IN ('OXACILLIN', 'CEFAZOLIN', 'CEFTRIAXONE', 'CEFEPIME') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS betalactam_r,
            -- 마크로라이드계: 에리스로마이신
            MAX(CASE WHEN ab_name IN ('ERYTHROMYCIN', 'AZITHROMYCIN', 'CLARITHROMYCIN') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS macrolide_r,
            -- 퀴놀론계: 레보플록사신
            MAX(CASE WHEN ab_name IN ('LEVOFLOXACIN', 'CIPROFLOXACIN', 'MOXIFLOXACIN') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS quinolone_r
        FROM microbiologyevents
        WHERE org_name = 'STAPH AUREUS COAG +'
        GROUP BY hadm_id
    )
    WHERE betalactam_r = 1 AND macrolide_r = 1 AND quinolone_r = 1
),
cre_cases AS (
    -- CRE: 카바페넴계(필수) + 세팔로스포린계 + 퀴놀론계 (3계열 모두 내성)
    SELECT hadm_id, 'CRE' AS mdro_type
    FROM (
        SELECT 
            hadm_id,
            -- 카바페넴계 (필수): 이미페넴, 메로페넴, 에르타페넴
            MAX(CASE WHEN ab_name IN ('IMIPENEM', 'MEROPENEM', 'ERTAPENEM') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS carbapenem_r,
            -- 세팔로스포린계: 세프트리아존
            MAX(CASE WHEN ab_name IN ('CEFTRIAXONE', 'CEFTAZIDIME', 'CEFEPIME') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS cephalosporin_r,
            -- 퀴놀론계: 시프로플록사신
            MAX(CASE WHEN ab_name IN ('CIPROFLOXACIN', 'LEVOFLOXACIN', 'MOXIFLOXACIN') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS quinolone_r
        FROM microbiologyevents
        WHERE org_name LIKE '%KLEBSIELLA%' 
           OR org_name LIKE '%ESCHERICHIA%' 
           OR org_name LIKE '%ENTEROBACTER%'
        GROUP BY hadm_id
    )
    WHERE carbapenem_r = 1 AND cephalosporin_r = 1 AND quinolone_r = 1
),
all_mdro AS (
    SELECT hadm_id, mdro_type FROM mrsa_cases
    UNION
    SELECT hadm_id, mdro_type FROM cre_cases
)

-- 입원별 MDRO 플래그 집계 + 나이 50세 이상 필터링
SELECT 
    m.hadm_id,
    STRING_AGG(DISTINCT m.mdro_type, ', ' ORDER BY m.mdro_type) AS mdro_types,
    MAX(CASE WHEN m.mdro_type = 'MRSA' THEN 1 ELSE 0 END) AS has_mrsa,
    MAX(CASE WHEN m.mdro_type = 'CRE' THEN 1 ELSE 0 END) AS has_cre,
    1 AS has_mdro
FROM all_mdro m
JOIN admissions a ON m.hadm_id = a.hadm_id
JOIN patients p ON a.subject_id = p.subject_id
WHERE (
    EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
    - CAST(p.anchor_year AS INTEGER) 
    + CAST(p.anchor_age AS INTEGER)
) >= 50
-- 체류 기간 7일 이상
AND (
    CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP)
) >= INTERVAL '7 days'
-- ICU 미경험 & 일반 병동
AND m.hadm_id IN (
    SELECT DISTINCT hadm_id 
    FROM transfers 
    WHERE careunit IN (
        'Medicine',
        'Med/Surg',
        'Medicine/Cardiology',
        'Neurology',
        'Transplant',
        'Hematology/Oncology',
        'Vascular',
        'Surgery/Trauma',
        'Med/Surg/GYN',
        'Med/Surg/Trauma',
        'Surgery',
        'Cardiac Surgery',
        'Surgery/Pancreatic/Biliary/Bariatric',
        'Surgery/Vascular/Intermediate',
        'Thoracic Surgery'
    )
)
GROUP BY m.hadm_id;
"""

conn.execute(mdro_query)

# MDRO 통계 확인
mdro_stats = conn.execute("""
    SELECT 
        COUNT(DISTINCT hadm_id) AS total_mdro_admissions,
        SUM(has_mrsa) AS mrsa_count,
        SUM(has_cre) AS cre_count
    FROM mdro_admissions
""").df()

print(f"  ✅ MDRO 테이블 생성 완료 (MRSA, CRE - 3계열 동반, 50세 이상)")
print(f"\n  MDRO 입원 현황:")
print(f"    전체 MDRO 입원: {mdro_stats['total_mdro_admissions'].iloc[0]:,}건")
print(f"    MRSA (3계열): {mdro_stats['mrsa_count'].iloc[0]:,}건")
print(f"    CRE (3계열): {mdro_stats['cre_count'].iloc[0]:,}건")


[Step 1] microbiologyevents에서 MDRO 추출 (MRSA, CRE - 3계열 동반)...
  ✅ MDRO 테이블 생성 완료 (MRSA, CRE - 3계열 동반, 50세 이상)

  MDRO 입원 현황:
    전체 MDRO 입원: 1,548건
    MRSA (3계열): 1,507.0건
    CRE (3계열): 44.0건


### 2) MRSA/CRE 환자 진단 분포

In [23]:
# MRSA 환자 진단 분포
mrsa_dx = conn.execute("""
    SELECT 
        d.icd_code,
        d.icd_version,
        di.long_title,
        COUNT(*) as cnt
    FROM mdro_admissions m
    JOIN diagnoses_icd d ON m.hadm_id = d.hadm_id
    JOIN d_icd_diagnoses di ON d.icd_code = di.icd_code AND d.icd_version = di.icd_version
    WHERE m.has_mrsa = 1
      AND d.seq_num = 1  -- 주진단만
    GROUP BY d.icd_code, d.icd_version, di.long_title
    ORDER BY cnt DESC
    LIMIT 20
""").df()

print("=== MRSA 환자 주진단 TOP 20 ===")
print(mrsa_dx.to_string())

# CRE 환자 진단 분포
cre_dx = conn.execute("""
    SELECT 
        d.icd_code,
        d.icd_version,
        di.long_title,
        COUNT(*) as cnt
    FROM mdro_admissions m
    JOIN diagnoses_icd d ON m.hadm_id = d.hadm_id
    JOIN d_icd_diagnoses di ON d.icd_code = di.icd_code AND d.icd_version = di.icd_version
    WHERE m.has_cre = 1
      AND d.seq_num = 1  -- 주진단만
    GROUP BY d.icd_code, d.icd_version, di.long_title
    ORDER BY cnt DESC
    LIMIT 20
""").df()

print("\n=== CRE 환자 주진단 TOP 20 ===")
print(cre_dx.to_string())

=== MRSA 환자 주진단 TOP 20 ===
   icd_code icd_version                                                                                                             long_title  cnt
0     99859           9                                                                                          Other postoperative infection   46
1      0389           9                                                                                                 Unspecified septicemia   42
2     A4102          10                                                              Sepsis due to Methicillin resistant Staphylococcus aureus   40
3      A419          10                                                                                           Sepsis, unspecified organism   39
4     03812           9                                                                 Methicillin resistant Staphylococcus aureus septicemia   37
5     44024           9                                                    Atheroscle

### +) 입원 경로, 성별, 나이 분포도 보기

In [18]:
# 입원 경로, 성별, 나이 분포도 보기
demo_stats = conn.execute("""
    SELECT 
        m.mdro_types,
        a.admission_type,
        p.gender,
        COUNT(*) as cnt,
        AVG(EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
            - CAST(p.anchor_year AS INTEGER) 
            + CAST(p.anchor_age AS INTEGER)) as avg_age
    FROM mdro_admissions m
    JOIN admissions a ON m.hadm_id = a.hadm_id
    JOIN patients p ON a.subject_id = p.subject_id
    GROUP BY m.mdro_types, a.admission_type, p.gender
    ORDER BY m.mdro_types, cnt DESC
""").df()

print("\n=== MDRO별 입원유형/성별 분포 ===")
print(demo_stats.to_string())


=== MDRO별 입원유형/성별 분포 ===
   mdro_types               admission_type gender  cnt    avg_age
0         CRE                     EW EMER.      M   14  68.428571
1         CRE                     EW EMER.      F    9  74.000000
2         CRE                 DIRECT EMER.      M    4  73.500000
3         CRE                       URGENT      M    4  57.750000
4         CRE            OBSERVATION ADMIT      M    4  71.000000
5         CRE                       URGENT      F    3  64.333333
6         CRE  SURGICAL SAME DAY ADMISSION      M    1  67.000000
7         CRE            OBSERVATION ADMIT      F    1  62.000000
8         CRE                 DIRECT EMER.      F    1  79.000000
9   CRE, MRSA                       URGENT      F    1  68.000000
10  CRE, MRSA            OBSERVATION ADMIT      M    1  55.000000
11  CRE, MRSA  SURGICAL SAME DAY ADMISSION      M    1  66.000000
12       MRSA                     EW EMER.      M  440  68.893182
13       MRSA                     EW EMER.      F 

### 3) MRSA/CRE 대표 환자 추출

In [19]:
# MRSA - 진단별 2명씩, 연령대 중복 없이
mrsa_candidates = conn.execute("""
WITH mrsa_with_age AS (
    SELECT 
        m.hadm_id,
        a.subject_id,
        d.icd_code,
        di.long_title,
        p.gender,
        (EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
         - CAST(p.anchor_year AS INTEGER) 
         + CAST(p.anchor_age AS INTEGER)) AS age,
        FLOOR((EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
         - CAST(p.anchor_year AS INTEGER) 
         + CAST(p.anchor_age AS INTEGER)) / 10) * 10 AS age_group,
        CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP) AS los
    FROM mdro_admissions m
    JOIN admissions a ON m.hadm_id = a.hadm_id
    JOIN patients p ON a.subject_id = p.subject_id
    JOIN diagnoses_icd d ON m.hadm_id = d.hadm_id AND d.seq_num = 1
    JOIN d_icd_diagnoses di ON d.icd_code = di.icd_code AND d.icd_version = di.icd_version
    WHERE m.has_mrsa = 1
      AND d.icd_code IN ('99859', '03812', '48242', '25080')
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY icd_code, age_group ORDER BY los DESC) as rn_in_age,
        ROW_NUMBER() OVER (PARTITION BY icd_code ORDER BY age_group, los DESC) as rn_overall
    FROM mrsa_with_age
)
SELECT 
    hadm_id,
    subject_id,
    icd_code,
    long_title,
    gender,
    age,
    age_group,
    los
FROM ranked
WHERE rn_in_age = 1  -- 연령대별 1명씩만
ORDER BY icd_code, age_group
""").df()

print("=== MRSA 후보 (진단별, 연령대별 1명씩) ===")
print(mrsa_candidates.to_string())

# 진단별로 연령대 다른 2명씩 최종 선택
mrsa_final = conn.execute("""
WITH mrsa_with_age AS (
    SELECT 
        m.hadm_id,
        a.subject_id,
        d.icd_code,
        di.long_title,
        p.gender,
        (EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
         - CAST(p.anchor_year AS INTEGER) 
         + CAST(p.anchor_age AS INTEGER)) AS age,
        FLOOR((EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
         - CAST(p.anchor_year AS INTEGER) 
         + CAST(p.anchor_age AS INTEGER)) / 10) * 10 AS age_group,
        CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP) AS los
    FROM mdro_admissions m
    JOIN admissions a ON m.hadm_id = a.hadm_id
    JOIN patients p ON a.subject_id = p.subject_id
    JOIN diagnoses_icd d ON m.hadm_id = d.hadm_id AND d.seq_num = 1
    JOIN d_icd_diagnoses di ON d.icd_code = di.icd_code AND d.icd_version = di.icd_version
    WHERE m.has_mrsa = 1
      AND d.icd_code IN ('99859', '03812', '48242', '25080')
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY icd_code, age_group ORDER BY los DESC) as rn_in_age
    FROM mrsa_with_age
),
unique_age AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY icd_code ORDER BY age_group) as age_rank
    FROM ranked
    WHERE rn_in_age = 1
)
SELECT 
    hadm_id,
    subject_id,
    icd_code,
    long_title,
    gender,
    age,
    age_group,
    los
FROM unique_age
WHERE age_rank <= 2  -- 진단별 2명 (연령대 다르게)
ORDER BY icd_code, age_group
""").df()

print("\n=== MRSA 최종 선택 (진단별 2명, 연령대 중복 없음) ===")
print(mrsa_final.to_string())

=== MRSA 후보 (진단별, 연령대별 1명씩) ===
     hadm_id subject_id icd_code                                                                                             long_title gender  age  age_group               los
0   22875063   13938358    03812                                                 Methicillin resistant Staphylococcus aureus septicemia      M   52       50.0  22 days 02:19:00
1   27165344   19432448    03812                                                 Methicillin resistant Staphylococcus aureus septicemia      M   64       60.0  27 days 19:14:00
2   27154619   16126164    03812                                                 Methicillin resistant Staphylococcus aureus septicemia      M   75       70.0 106 days 19:58:00
3   26115901   17636749    03812                                                 Methicillin resistant Staphylococcus aureus septicemia      F   81       80.0  58 days 21:06:00
4   26416160   16526239    03812                                                 Me

In [20]:
# CRE - 해당 진단 전부
cre_final = conn.execute("""
SELECT 
    m.hadm_id,
    a.subject_id,
    d.icd_code,
    di.long_title,
    p.gender,
    (EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
     - CAST(p.anchor_year AS INTEGER) 
     + CAST(p.anchor_age AS INTEGER)) AS age,
    CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP) AS los
FROM mdro_admissions m
JOIN admissions a ON m.hadm_id = a.hadm_id
JOIN patients p ON a.subject_id = p.subject_id
JOIN diagnoses_icd d ON m.hadm_id = d.hadm_id AND d.seq_num = 1
JOIN d_icd_diagnoses di ON d.icd_code = di.icd_code AND d.icd_version = di.icd_version
WHERE m.has_cre = 1
  AND d.icd_code IN ('5770', '57400', '5712')
ORDER BY d.icd_code
""").df()

print("\n=== CRE 최종 선택 ===")
print(cre_final.to_string())


=== CRE 최종 선택 ===
    hadm_id subject_id icd_code                                                                        long_title gender  age               los
0  28029495   16121672     5712                                                      Alcoholic cirrhosis of liver      F   61  12 days 00:22:00
1  22456860   18294629    57400  Calculus of gallbladder with acute cholecystitis, without mention of obstruction      M   69  11 days 23:14:00
2  28540703   12808803     5770                                                                Acute pancreatitis      M   76  44 days 20:21:00
3  25338284   19970491     5770                                                                Acute pancreatitis      M   55 133 days 02:18:00


---

## 2. Infection 대표 환자 추출
- Pneumonia
- GI_Infection
- UTI
- Skin_Infection

### 1) 감염 유형별 입원 시점 vs 원내 감염 분류 (Pneumonia, GI_Infection, UTI)

In [28]:
infection_classify_query = """
WITH infection_diagnosis AS (
    SELECT 
        d.hadm_id,
        d.icd_code,
        d.seq_num,
        CASE 
            -- Pneumonia
            WHEN d.icd_code LIKE 'J12%' OR d.icd_code LIKE 'J13%' 
                 OR d.icd_code LIKE 'J14%' OR d.icd_code LIKE 'J15%'
                 OR d.icd_code LIKE 'J16%' OR d.icd_code LIKE 'J17%' 
                 OR d.icd_code LIKE 'J18%'
                 OR d.icd_code LIKE '480%' OR d.icd_code LIKE '481%'
                 OR d.icd_code LIKE '482%' OR d.icd_code LIKE '483%'
                 OR d.icd_code LIKE '484%' OR d.icd_code LIKE '485%'
                 OR d.icd_code LIKE '486%' OR d.icd_code LIKE '487%'
            THEN 'Pneumonia'
            -- GI_Infection
            WHEN d.icd_code LIKE 'A0%' OR d.icd_code LIKE 'A09%'
                 OR d.icd_code LIKE 'K52%'
                 OR d.icd_code LIKE '001%' OR d.icd_code LIKE '002%'
                 OR d.icd_code LIKE '003%' OR d.icd_code LIKE '004%'
                 OR d.icd_code LIKE '005%' OR d.icd_code LIKE '006%'
                 OR d.icd_code LIKE '007%' OR d.icd_code LIKE '008%'
                 OR d.icd_code LIKE '009%' OR d.icd_code LIKE '5589%'
            THEN 'GI_Infection'
            -- UTI
            WHEN d.icd_code LIKE 'N390%' OR d.icd_code LIKE 'N391%'
                 OR d.icd_code LIKE 'N393%' OR d.icd_code LIKE 'N394%'
                 OR d.icd_code LIKE '5990%' OR d.icd_code LIKE '5997%'
            THEN 'UTI'
            ELSE NULL
        END AS infection_type
    FROM diagnoses_icd d
),
-- 입원별 감염 분류 (주진단 vs 부진단)
admission_infection_type AS (
    SELECT 
        hadm_id,
        infection_type,
        MIN(seq_num) AS min_seq_num,
        CASE 
            WHEN MIN(seq_num) = 1 THEN 'Admission'  -- 입원 당시 진단
            ELSE 'Hospital_Acquired'                 -- 원내 감염
        END AS infection_timing
    FROM infection_diagnosis
    WHERE infection_type IS NOT NULL
    GROUP BY hadm_id, infection_type
),
-- 50세 이상, 7일 이상 입원, 일반 병동 필터링
filtered_admissions AS (
    SELECT 
        a.hadm_id,
        a.subject_id,
        a.admittime,
        a.dischtime,
        (EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
         - CAST(p.anchor_year AS INTEGER) 
         + CAST(p.anchor_age AS INTEGER)) AS age,
        p.gender,
        CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP) AS los
    FROM admissions a
    JOIN patients p ON a.subject_id = p.subject_id
    WHERE (
        EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
        - CAST(p.anchor_year AS INTEGER) 
        + CAST(p.anchor_age AS INTEGER)
    ) >= 50
    AND (
        CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP)
    ) >= INTERVAL '7 days'
    AND a.hadm_id NOT IN (SELECT DISTINCT hadm_id FROM icustays WHERE hadm_id IS NOT NULL)
    AND a.hadm_id IN (
        SELECT DISTINCT hadm_id 
        FROM transfers 
        WHERE careunit IN (
            'Medicine',
            'Med/Surg',
            'Medicine/Cardiology',
            'Neurology',
            'Transplant',
            'Hematology/Oncology',
            'Vascular',
            'Surgery/Trauma',
            'Med/Surg/GYN',
            'Med/Surg/Trauma',
            'Surgery',
            'Cardiac Surgery',
            'Surgery/Pancreatic/Biliary/Bariatric',
            'Surgery/Vascular/Intermediate',
            'Thoracic Surgery'
        )
    )
)
-- 최종 집계
SELECT 
    ait.infection_type,
    ait.infection_timing,
    COUNT(DISTINCT ait.hadm_id) AS admission_count,
    COUNT(DISTINCT fa.subject_id) AS patient_count,
    ROUND(AVG(fa.age), 1) AS avg_age,
    ROUND(AVG(EXTRACT(EPOCH FROM fa.los) / 86400), 1) AS avg_los_days
FROM admission_infection_type ait
JOIN filtered_admissions fa ON ait.hadm_id = fa.hadm_id
GROUP BY ait.infection_type, ait.infection_timing
ORDER BY ait.infection_type, ait.infection_timing
"""

infection_stats = conn.execute(infection_classify_query).df()

print("\n=== 감염 유형별 입원시점 vs 원내감염 분포 (50세↑, 7일↑, 일반 병동) ===")
print(infection_stats.to_string(index=False))


=== 감염 유형별 입원시점 vs 원내감염 분포 (50세↑, 7일↑, 일반 병동) ===
infection_type  infection_timing  admission_count  patient_count  avg_age  avg_los_days
  GI_Infection         Admission              415            400     70.8          12.0
  GI_Infection Hospital_Acquired             2046           1859     70.5          16.7
     Pneumonia         Admission              617            588     72.6          11.4
     Pneumonia Hospital_Acquired             3130           2883     71.7          15.9
           UTI         Admission              395            376     75.5          12.7
           UTI Hospital_Acquired             5611           4960     73.7          14.4


In [29]:
print("\n=== 요약 테이블 ===")
pivot = infection_stats.pivot(index='infection_type', columns='infection_timing', values='admission_count').fillna(0)
pivot['Total'] = pivot.sum(axis=1)
print(pivot)


=== 요약 테이블 ===
infection_timing  Admission  Hospital_Acquired  Total
infection_type                                       
GI_Infection            415               2046   2461
Pneumonia               617               3130   3747
UTI                     395               5611   6006


### 2) 추가 필터링: 항생제 처방 + 배양 기록

In [32]:
print("\n[추가 필터링] 항생제 처방 + 배양 기록 있는 환자...")

# 먼저 기존 필터 조건에 맞는 hadm_id 리스트 추출
base_cohort_query = """
WITH infection_diagnosis AS (
    SELECT 
        d.hadm_id,
        d.icd_code,
        d.seq_num,
        CASE 
            WHEN d.icd_code LIKE 'J12%' OR d.icd_code LIKE 'J13%' 
                 OR d.icd_code LIKE 'J14%' OR d.icd_code LIKE 'J15%'
                 OR d.icd_code LIKE 'J16%' OR d.icd_code LIKE 'J17%' 
                 OR d.icd_code LIKE 'J18%'
                 OR d.icd_code LIKE '480%' OR d.icd_code LIKE '481%'
                 OR d.icd_code LIKE '482%' OR d.icd_code LIKE '483%'
                 OR d.icd_code LIKE '484%' OR d.icd_code LIKE '485%'
                 OR d.icd_code LIKE '486%' OR d.icd_code LIKE '487%'
            THEN 'Pneumonia'
            WHEN d.icd_code LIKE 'A0%' OR d.icd_code LIKE 'A09%'
                 OR d.icd_code LIKE 'K52%'
                 OR d.icd_code LIKE '001%' OR d.icd_code LIKE '002%'
                 OR d.icd_code LIKE '003%' OR d.icd_code LIKE '004%'
                 OR d.icd_code LIKE '005%' OR d.icd_code LIKE '006%'
                 OR d.icd_code LIKE '007%' OR d.icd_code LIKE '008%'
                 OR d.icd_code LIKE '009%' OR d.icd_code LIKE '5589%'
            THEN 'GI_Infection'
            WHEN d.icd_code LIKE 'N390%' OR d.icd_code LIKE 'N391%'
                 OR d.icd_code LIKE 'N393%' OR d.icd_code LIKE 'N394%'
                 OR d.icd_code LIKE '5990%' OR d.icd_code LIKE '5997%'
            THEN 'UTI'
            ELSE NULL
        END AS infection_type
    FROM diagnoses_icd d
),
admission_infection_type AS (
    SELECT 
        hadm_id,
        infection_type,
        MIN(seq_num) AS min_seq_num,
        CASE 
            WHEN MIN(seq_num) = 1 THEN 'Admission'
            ELSE 'Hospital_Acquired'
        END AS infection_timing
    FROM infection_diagnosis
    WHERE infection_type IS NOT NULL
    GROUP BY hadm_id, infection_type
),
filtered_admissions AS (
    SELECT 
        a.hadm_id,
        a.subject_id,
        (EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
         - CAST(p.anchor_year AS INTEGER) 
         + CAST(p.anchor_age AS INTEGER)) AS age,
        p.gender,
        EXTRACT(EPOCH FROM (CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP))) / 86400 AS los_days
    FROM admissions a
    JOIN patients p ON a.subject_id = p.subject_id
    WHERE (
        EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
        - CAST(p.anchor_year AS INTEGER) 
        + CAST(p.anchor_age AS INTEGER)
    ) >= 50
    AND (
        CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP)
    ) >= INTERVAL '7 days'
    AND a.hadm_id NOT IN (SELECT DISTINCT hadm_id FROM icustays WHERE hadm_id IS NOT NULL)
    AND a.hadm_id IN (
        SELECT DISTINCT hadm_id 
        FROM transfers 
        WHERE careunit IN (
            'Medicine', 'Med/Surg', 'Medicine/Cardiology', 'Neurology',
            'Transplant', 'Hematology/Oncology', 'Vascular', 'Surgery/Trauma',
            'Med/Surg/GYN', 'Med/Surg/Trauma', 'Surgery', 'Cardiac Surgery',
            'Surgery/Pancreatic/Biliary/Bariatric', 'Surgery/Vascular/Intermediate',
            'Thoracic Surgery'
        )
    )
),
-- 항생제 처방 횟수
abx_counts AS (
    SELECT 
        hadm_id,
        COUNT(DISTINCT pharmacy_id) AS abx_count
    FROM pharmacy
    WHERE LOWER(medication) LIKE '%cillin%'
       OR LOWER(medication) LIKE '%mycin%'
       OR LOWER(medication) LIKE '%floxacin%'
       OR LOWER(medication) LIKE '%cycline%'
       OR LOWER(medication) LIKE '%sulfa%'
       OR LOWER(medication) LIKE '%metronidazole%'
       OR LOWER(medication) LIKE '%vancomycin%'
       OR LOWER(medication) LIKE '%meropenem%'
       OR LOWER(medication) LIKE '%imipenem%'
       OR LOWER(medication) LIKE '%cef%'
       OR LOWER(medication) LIKE '%penem%'
    GROUP BY hadm_id
),
-- 배양 기록 횟수
culture_counts AS (
    SELECT 
        hadm_id,
        COUNT(DISTINCT micro_specimen_id) AS culture_count
    FROM microbiologyevents
    WHERE hadm_id IS NOT NULL
    GROUP BY hadm_id
)
SELECT 
    ait.infection_type,
    ait.infection_timing,
    ait.hadm_id,
    fa.subject_id,
    fa.age,
    fa.gender,
    fa.los_days,
    ac.abx_count,
    cc.culture_count
FROM admission_infection_type ait
JOIN filtered_admissions fa ON ait.hadm_id = fa.hadm_id
JOIN abx_counts ac ON ait.hadm_id = ac.hadm_id
JOIN culture_counts cc ON ait.hadm_id = cc.hadm_id
"""

cohort_detailed = conn.execute(base_cohort_query).df()

print(f"✅ 필터링 완료: {len(cohort_detailed):,}건")
print(f"   Unique patients: {cohort_detailed['subject_id'].nunique():,}명")


[추가 필터링] 항생제 처방 + 배양 기록 있는 환자...
✅ 필터링 완료: 10,624건
   Unique patients: 7,805명


In [33]:
# ============================================
# 통계 요약 (avg, min, max, median)
# ============================================

print("\n=== 감염 유형별 상세 통계 ===\n")

stats_summary = cohort_detailed.groupby(['infection_type', 'infection_timing']).agg(
    admission_count=('hadm_id', 'nunique'),
    patient_count=('subject_id', 'nunique'),
    # 나이
    age_avg=('age', 'mean'),
    age_min=('age', 'min'),
    age_max=('age', 'max'),
    age_median=('age', 'median'),
    # 재원일수
    los_avg=('los_days', 'mean'),
    los_min=('los_days', 'min'),
    los_max=('los_days', 'max'),
    los_median=('los_days', 'median'),
    # 항생제 처방
    abx_avg=('abx_count', 'mean'),
    abx_min=('abx_count', 'min'),
    abx_max=('abx_count', 'max'),
    abx_median=('abx_count', 'median'),
    # 배양 기록
    culture_avg=('culture_count', 'mean'),
    culture_min=('culture_count', 'min'),
    culture_max=('culture_count', 'max'),
    culture_median=('culture_count', 'median'),
).round(1).reset_index()

print(stats_summary.to_string(index=False))


=== 감염 유형별 상세 통계 ===

infection_type  infection_timing  admission_count  patient_count  age_avg  age_min  age_max  age_median  los_avg  los_min  los_max  los_median  abx_avg  abx_min  abx_max  abx_median  culture_avg  culture_min  culture_max  culture_median
  GI_Infection         Admission              367            353     70.6       50       98        70.0     12.2      7.0     92.0         9.9      8.2        1       42         7.0          6.0            1           29             5.0
  GI_Infection Hospital_Acquired             1796           1651     70.3       50      100        70.0     17.2      7.0    234.0        12.8     10.4        1       51         9.0          7.8            1           75             5.0
     Pneumonia         Admission              576            548     72.2       50       98        72.0     11.5      7.0     60.2         9.7      8.7        1       38         8.0          6.4            1           42             5.0
     Pneumonia Hospital_Acqui

In [34]:
# ============================================
# 피벗 요약
# ============================================

print("\n=== 입원 수 요약 (Admission vs Hospital_Acquired) ===")
pivot = cohort_detailed.groupby(['infection_type', 'infection_timing'])['hadm_id'].nunique().unstack(fill_value=0)
pivot['Total'] = pivot.sum(axis=1)
print(pivot)


=== 입원 수 요약 (Admission vs Hospital_Acquired) ===
infection_timing  Admission  Hospital_Acquired  Total
infection_type                                       
GI_Infection            367               1796   2163
Pneumonia               576               2778   3354
UTI                     312               4795   5107


### 3) 추가 필터링: 배양 종류 + 항생제 계열 상세 분포

In [43]:
print("\n[상세 분포] 배양 종류 + 항생제 계열...")

# cohort_detailed에서 hadm_id 추출해서 임시 테이블 생성 (INTEGER로 캐스팅)
hadm_list = cohort_detailed['hadm_id'].unique().tolist()
conn.execute(f"""
    CREATE OR REPLACE TEMP TABLE cohort_detailed_temp AS 
    SELECT CAST(hadm_id AS INTEGER) AS hadm_id 
    FROM (VALUES {','.join([f'({x})' for x in hadm_list])}) AS t(hadm_id)
""")

print(f"✅ 임시 테이블 생성: {len(hadm_list):,}건")


[상세 분포] 배양 종류 + 항생제 계열...
✅ 임시 테이블 생성: 9,331건


In [45]:
# 1. 배양 검체 종류 분포
culture_dist = conn.execute("""
SELECT 
    m.spec_type_desc AS specimen_type,
    COUNT(*) AS total_cnt,
    COUNT(DISTINCT m.hadm_id) AS admission_cnt
FROM microbiologyevents m
WHERE CAST(m.hadm_id AS INTEGER) IN (SELECT hadm_id FROM cohort_detailed_temp)
GROUP BY m.spec_type_desc
ORDER BY total_cnt DESC
LIMIT 20
""").df()

print("\n=== 배양 검체 종류 TOP 20 ===")
print(culture_dist.to_string(index=False))


=== 배양 검체 종류 TOP 20 ===
                           specimen_type  total_cnt  admission_cnt
                                   URINE      42028           6743
                           BLOOD CULTURE      33072           6036
                                   STOOL      11480           3343
                                  SPUTUM      11313           1741
                                  TISSUE       6709            537
                                    SWAB       6362            818
                  BRONCHOALVEOLAR LAVAGE       3841            352
                           PLEURAL FLUID       3236            541
                                 ABSCESS       2465            216
                        PERITONEAL FLUID       2289            421
                             MRSA SCREEN       1664           1247
Rapid Respiratory Viral Screen & Culture       1329            487
                        CSF;SPINAL FLUID       1235            306
                          SEROLOGY/BL

In [48]:
# 2. 항생제 계열 분포
abx_class_dist = conn.execute("""
SELECT 
    CASE 
        WHEN LOWER(medication) LIKE '%vancomycin%' THEN 'Glycopeptide (Vancomycin)'
        WHEN LOWER(medication) LIKE '%meropenem%' OR LOWER(medication) LIKE '%imipenem%' 
             OR LOWER(medication) LIKE '%ertapenem%' OR LOWER(medication) LIKE '%penem%' THEN 'Carbapenem'
        WHEN LOWER(medication) LIKE '%ceftriaxone%' OR LOWER(medication) LIKE '%cefazolin%' 
             OR LOWER(medication) LIKE '%cefepime%' OR LOWER(medication) LIKE '%ceftazidime%'
             OR LOWER(medication) LIKE '%cef%' THEN 'Cephalosporin'
        WHEN LOWER(medication) LIKE '%piperacillin%' OR LOWER(medication) LIKE '%ampicillin%'
             OR LOWER(medication) LIKE '%amoxicillin%' OR LOWER(medication) LIKE '%cillin%' THEN 'Penicillin'
        WHEN LOWER(medication) LIKE '%ciprofloxacin%' OR LOWER(medication) LIKE '%levofloxacin%'
             OR LOWER(medication) LIKE '%moxifloxacin%' OR LOWER(medication) LIKE '%floxacin%' THEN 'Fluoroquinolone'
        WHEN LOWER(medication) LIKE '%azithromycin%' OR LOWER(medication) LIKE '%erythromycin%'
             OR LOWER(medication) LIKE '%clarithromycin%' THEN 'Macrolide'
        WHEN LOWER(medication) LIKE '%gentamicin%' OR LOWER(medication) LIKE '%tobramycin%'
             OR LOWER(medication) LIKE '%amikacin%' THEN 'Aminoglycoside'
        WHEN LOWER(medication) LIKE '%doxycycline%' OR LOWER(medication) LIKE '%tetracycline%'
             OR LOWER(medication) LIKE '%minocycline%' OR LOWER(medication) LIKE '%cycline%' THEN 'Tetracycline'
        WHEN LOWER(medication) LIKE '%metronidazole%' THEN 'Nitroimidazole (Metronidazole)'
        WHEN LOWER(medication) LIKE '%sulfamethoxazole%' OR LOWER(medication) LIKE '%trimethoprim%'
             OR LOWER(medication) LIKE '%sulfa%' THEN 'Sulfonamide'
        WHEN LOWER(medication) LIKE '%linezolid%' THEN 'Oxazolidinone (Linezolid)'
        WHEN LOWER(medication) LIKE '%daptomycin%' THEN 'Lipopeptide (Daptomycin)'
        ELSE 'Other'
    END AS abx_class,
    COUNT(*) AS prescription_cnt,
    COUNT(DISTINCT hadm_id) AS admission_cnt
FROM pharmacy
WHERE CAST(hadm_id AS INTEGER) IN (SELECT hadm_id FROM cohort_detailed_temp)
  AND (LOWER(medication) LIKE '%cillin%'
       OR LOWER(medication) LIKE '%mycin%'
       OR LOWER(medication) LIKE '%floxacin%'
       OR LOWER(medication) LIKE '%cycline%'
       OR LOWER(medication) LIKE '%sulfa%'
       OR LOWER(medication) LIKE '%metronidazole%'
       OR LOWER(medication) LIKE '%vancomycin%'
       OR LOWER(medication) LIKE '%meropenem%'
       OR LOWER(medication) LIKE '%imipenem%'
       OR LOWER(medication) LIKE '%cef%'
       OR LOWER(medication) LIKE '%penem%'
       OR LOWER(medication) LIKE '%linezolid%'
       OR LOWER(medication) LIKE '%daptomycin%')
GROUP BY abx_class
ORDER BY prescription_cnt DESC
""").df()

print("\n=== 항생제 계열별 분포 ===")
print(abx_class_dist.to_string(index=False))


=== 항생제 계열별 분포 ===
                     abx_class  prescription_cnt  admission_cnt
                   Sulfonamide             28056           7367
     Glycopeptide (Vancomycin)             15527           5162
                 Cephalosporin             12722           6065
               Fluoroquinolone              7276           4094
                    Penicillin              4670           2318
Nitroimidazole (Metronidazole)              4286           2502
                     Macrolide              2581           1466
                    Carbapenem              2133            954
                         Other              1269            486
      Lipopeptide (Daptomycin)               676            302
                  Tetracycline               676            453
     Oxazolidinone (Linezolid)               648            350
                Aminoglycoside               104             55


In [ ]:
# 3. 감염 유형별 항생제 계열 크로스탭
infection_abx_cross = conn.execute("""
WITH abx_classified AS (
    SELECT 
        hadm_id,
        CASE 
            WHEN LOWER(medication) LIKE '%vancomycin%' THEN 'Glycopeptide'
            WHEN LOWER(medication) LIKE '%penem%' THEN 'Carbapenem'
            WHEN LOWER(medication) LIKE '%cef%' THEN 'Cephalosporin'
            WHEN LOWER(medication) LIKE '%cillin%' THEN 'Penicillin'
            WHEN LOWER(medication) LIKE '%floxacin%' THEN 'Fluoroquinolone'
            WHEN LOWER(medication) LIKE '%azithromycin%' OR LOWER(medication) LIKE '%erythromycin%' THEN 'Macrolide'
            WHEN LOWER(medication) LIKE '%gentamicin%' OR LOWER(medication) LIKE '%tobramycin%' THEN 'Aminoglycoside'
            WHEN LOWER(medication) LIKE '%metronidazole%' THEN 'Metronidazole'
            ELSE 'Other'
        END AS abx_class
    FROM pharmacy
    WHERE CAST(hadm_id AS INTEGER) IN (SELECT hadm_id FROM cohort_detailed_temp)
)
SELECT 
    cd.infection_type,
    ac.abx_class,
    COUNT(DISTINCT cd.hadm_id) AS admission_cnt
FROM cohort_detailed cd
JOIN abx_classified ac ON cd.hadm_id = ac.hadm_id
GROUP BY cd.infection_type, ac.abx_class
ORDER BY cd.infection_type, admission_cnt DESC
""")

# cohort_detailed를 DuckDB에서 직접 참조할 수 없으니 Python에서 처리
# 대신 cohort_detailed df를 활용
abx_by_hadm = conn.execute("""
SELECT 
    hadm_id,
    CASE 
        WHEN LOWER(medication) LIKE '%vancomycin%' THEN 'Glycopeptide'
        WHEN LOWER(medication) LIKE '%penem%' THEN 'Carbapenem'
        WHEN LOWER(medication) LIKE '%cef%' THEN 'Cephalosporin'
        WHEN LOWER(medication) LIKE '%cillin%' THEN 'Penicillin'
        WHEN LOWER(medication) LIKE '%floxacin%' THEN 'Fluoroquinolone'
        WHEN LOWER(medication) LIKE '%azithromycin%' OR LOWER(medication) LIKE '%erythromycin%' THEN 'Macrolide'
        WHEN LOWER(medication) LIKE '%gentamicin%' OR LOWER(medication) LIKE '%tobramycin%' THEN 'Aminoglycoside'
        WHEN LOWER(medication) LIKE '%metronidazole%' THEN 'Metronidazole'
        ELSE 'Other'
    END AS abx_class
FROM pharmacy
WHERE CAST(hadm_id AS INTEGER) IN (SELECT hadm_id FROM cohort_detailed_temp)
""").df()

# cohort_detailed와 조인
merged = cohort_detailed[['hadm_id', 'infection_type', 'infection_timing']].drop_duplicates().merge(
    abx_by_hadm, on='hadm_id'
)

print("\n=== 감염 유형별 항생제 계열 분포 ===")
infection_abx_summary = merged.groupby(['infection_type', 'abx_class'])['hadm_id'].nunique().reset_index()
infection_abx_summary.columns = ['infection_type', 'abx_class', 'admission_cnt']
infection_abx_summary = infection_abx_summary.sort_values(['infection_type', 'admission_cnt'], ascending=[True, False])
print(infection_abx_summary.to_string(index=False))


=== 감염 유형별 항생제 계열 분포 ===
infection_type       abx_class  admission_cnt
  GI_Infection           Other           2163
  GI_Infection    Glycopeptide           1545
  GI_Infection   Metronidazole           1240
  GI_Infection   Cephalosporin           1208
  GI_Infection Fluoroquinolone            769
  GI_Infection      Penicillin            521
  GI_Infection      Carbapenem            264
  GI_Infection       Macrolide            202
  GI_Infection  Aminoglycoside             36
     Pneumonia           Other           3354
     Pneumonia   Cephalosporin           2492
     Pneumonia    Glycopeptide           2315
     Pneumonia Fluoroquinolone           1650
     Pneumonia       Macrolide           1117
     Pneumonia      Penicillin            928
     Pneumonia   Metronidazole            856
     Pneumonia      Carbapenem            338
     Pneumonia  Aminoglycoside             53
           UTI           Other           5107
           UTI   Cephalosporin           3312
        

In [52]:
print("\n=== 감염 유형 × 항생제 계열 크로스탭 ===")
cross_pivot = infection_abx_summary.pivot(index='infection_type', columns='abx_class', values='admission_cnt').fillna(0).astype(int)
print(cross_pivot)


=== 감염 유형 × 항생제 계열 크로스탭 ===
abx_class       Aminoglycoside  Carbapenem  Cephalosporin  Fluoroquinolone  \
infection_type                                                               
GI_Infection                36         264           1208              769   
Pneumonia                   53         338           2492             1650   
UTI                         94         547           3312             2291   

abx_class       Glycopeptide  Macrolide  Metronidazole  Other  Penicillin  
infection_type                                                             
GI_Infection            1545        202           1240   2163         521  
Pneumonia               2315       1117            856   3354         928  
UTI                     2275        408            965   5107        1279  


In [53]:
# 4. 감염 유형별 배양 검체 분포
culture_by_hadm = conn.execute("""
SELECT 
    hadm_id,
    spec_type_desc AS specimen_type
FROM microbiologyevents
WHERE CAST(hadm_id AS INTEGER) IN (SELECT hadm_id FROM cohort_detailed_temp)
""").df()

merged_culture = cohort_detailed[['hadm_id', 'infection_type', 'infection_timing']].drop_duplicates().merge(
    culture_by_hadm, on='hadm_id'
)

print("\n=== 감염 유형별 배양 검체 분포 ===")
infection_culture_summary = merged_culture.groupby(['infection_type', 'specimen_type'])['hadm_id'].nunique().reset_index()
infection_culture_summary.columns = ['infection_type', 'specimen_type', 'admission_cnt']
infection_culture_summary = infection_culture_summary.sort_values(['infection_type', 'admission_cnt'], ascending=[True, False])

# 감염 유형별 TOP 5 검체만 출력
for inf_type in ['Pneumonia', 'GI_Infection', 'UTI']:
    print(f"\n[{inf_type}] TOP 5 검체:")
    subset = infection_culture_summary[infection_culture_summary['infection_type'] == inf_type].head(5)
    print(subset.to_string(index=False))


=== 감염 유형별 배양 검체 분포 ===

[Pneumonia] TOP 5 검체:
infection_type specimen_type  admission_cnt
     Pneumonia BLOOD CULTURE           2383
     Pneumonia         URINE           2329
     Pneumonia        SPUTUM           1439
     Pneumonia         STOOL           1015
     Pneumonia   MRSA SCREEN            897

[GI_Infection] TOP 5 검체:
infection_type specimen_type  admission_cnt
  GI_Infection         STOOL           1587
  GI_Infection BLOOD CULTURE           1411
  GI_Infection         URINE           1290
  GI_Infection        SPUTUM            248
  GI_Infection   MRSA SCREEN            226

[UTI] TOP 5 검체:
infection_type specimen_type  admission_cnt
           UTI         URINE           4167
           UTI BLOOD CULTURE           3185
           UTI         STOOL           1472
           UTI          SWAB            468
           UTI        SPUTUM            425


### 4) 대표 환자 추출

In [54]:
print("\n[대표 환자 추출] 배양 4종 필수 + 감염별 항생제 필수...")

representative_query = """
WITH -- 배양 4종 모두 있는 환자
culture_all_four AS (
    SELECT hadm_id
    FROM microbiologyevents
    WHERE spec_type_desc IN ('URINE', 'BLOOD CULTURE', 'STOOL', 'SPUTUM')
    GROUP BY hadm_id
    HAVING COUNT(DISTINCT spec_type_desc) = 4
),
-- 항생제 계열별 사용 여부
abx_by_class AS (
    SELECT 
        hadm_id,
        MAX(CASE WHEN LOWER(medication) LIKE '%metronidazole%' THEN 1 ELSE 0 END) AS has_metronidazole,
        MAX(CASE WHEN LOWER(medication) LIKE '%floxacin%' THEN 1 ELSE 0 END) AS has_fluoroquinolone,
        MAX(CASE WHEN LOWER(medication) LIKE '%cef%' THEN 1 ELSE 0 END) AS has_cephalosporin,
        MAX(CASE WHEN LOWER(medication) LIKE '%azithromycin%' 
                 OR LOWER(medication) LIKE '%erythromycin%' 
                 OR LOWER(medication) LIKE '%clarithromycin%' THEN 1 ELSE 0 END) AS has_macrolide
    FROM pharmacy
    GROUP BY hadm_id
),
-- 감염 진단 분류
infection_diagnosis AS (
    SELECT 
        hadm_id,
        CASE 
            WHEN icd_code LIKE 'J12%' OR icd_code LIKE 'J13%' 
                 OR icd_code LIKE 'J14%' OR icd_code LIKE 'J15%'
                 OR icd_code LIKE 'J16%' OR icd_code LIKE 'J17%' 
                 OR icd_code LIKE 'J18%'
                 OR icd_code LIKE '480%' OR icd_code LIKE '481%'
                 OR icd_code LIKE '482%' OR icd_code LIKE '483%'
                 OR icd_code LIKE '484%' OR icd_code LIKE '485%'
                 OR icd_code LIKE '486%' OR icd_code LIKE '487%'
            THEN 'Pneumonia'
            WHEN icd_code LIKE 'A0%' OR icd_code LIKE 'A09%'
                 OR icd_code LIKE 'K52%'
                 OR icd_code LIKE '001%' OR icd_code LIKE '002%'
                 OR icd_code LIKE '003%' OR icd_code LIKE '004%'
                 OR icd_code LIKE '005%' OR icd_code LIKE '006%'
                 OR icd_code LIKE '007%' OR icd_code LIKE '008%'
                 OR icd_code LIKE '009%' OR icd_code LIKE '5589%'
            THEN 'GI_Infection'
            WHEN icd_code LIKE 'N390%' OR icd_code LIKE 'N391%'
                 OR icd_code LIKE 'N393%' OR icd_code LIKE 'N394%'
                 OR icd_code LIKE '5990%' OR icd_code LIKE '5997%'
            THEN 'UTI'
            ELSE NULL
        END AS infection_type,
        seq_num
    FROM diagnoses_icd
),
admission_infection AS (
    SELECT 
        hadm_id,
        infection_type,
        CASE WHEN MIN(seq_num) = 1 THEN 'Admission' ELSE 'Hospital_Acquired' END AS infection_timing
    FROM infection_diagnosis
    WHERE infection_type IS NOT NULL
    GROUP BY hadm_id, infection_type
),
-- 기본 필터 (50세↑, 7일↑, ICU 미경험, 일반병동)
filtered_admissions AS (
    SELECT 
        a.hadm_id,
        a.subject_id,
        (EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
         - CAST(p.anchor_year AS INTEGER) 
         + CAST(p.anchor_age AS INTEGER)) AS age,
        p.gender,
        EXTRACT(EPOCH FROM (CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP))) / 86400 AS los_days
    FROM admissions a
    JOIN patients p ON a.subject_id = p.subject_id
    WHERE (
        EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
        - CAST(p.anchor_year AS INTEGER) 
        + CAST(p.anchor_age AS INTEGER)
    ) >= 50
    AND (
        CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP)
    ) >= INTERVAL '7 days'
    AND a.hadm_id NOT IN (SELECT DISTINCT hadm_id FROM icustays WHERE hadm_id IS NOT NULL)
    AND a.hadm_id IN (
        SELECT DISTINCT hadm_id 
        FROM transfers 
        WHERE careunit IN (
            'Medicine', 'Med/Surg', 'Medicine/Cardiology', 'Neurology',
            'Transplant', 'Hematology/Oncology', 'Vascular', 'Surgery/Trauma',
            'Med/Surg/GYN', 'Med/Surg/Trauma', 'Surgery', 'Cardiac Surgery',
            'Surgery/Pancreatic/Biliary/Bariatric', 'Surgery/Vascular/Intermediate',
            'Thoracic Surgery'
        )
    )
)
-- 최종: 감염별 항생제 조건 적용
SELECT 
    ai.infection_type,
    ai.infection_timing,
    fa.hadm_id,
    fa.subject_id,
    fa.age,
    fa.gender,
    fa.los_days,
    ab.has_metronidazole,
    ab.has_fluoroquinolone,
    ab.has_cephalosporin,
    ab.has_macrolide
FROM admission_infection ai
JOIN filtered_admissions fa ON ai.hadm_id = fa.hadm_id
JOIN culture_all_four cf ON ai.hadm_id = cf.hadm_id
JOIN abx_by_class ab ON ai.hadm_id = ab.hadm_id
WHERE 
    -- GI_Infection: Metronidazole + Fluoroquinolone 필수
    (ai.infection_type = 'GI_Infection' AND ab.has_metronidazole = 1 AND ab.has_fluoroquinolone = 1)
    OR
    -- Pneumonia: Cephalosporin + Fluoroquinolone + Macrolide 필수
    (ai.infection_type = 'Pneumonia' AND ab.has_cephalosporin = 1 AND ab.has_fluoroquinolone = 1 AND ab.has_macrolide = 1)
    OR
    -- UTI: Fluoroquinolone + Metronidazole 필수
    (ai.infection_type = 'UTI' AND ab.has_fluoroquinolone = 1 AND ab.has_metronidazole = 1)
"""

representative_df = conn.execute(representative_query).df()

print(f"✅ 대표 환자 후보 추출 완료")


[대표 환자 추출] 배양 4종 필수 + 감염별 항생제 필수...
✅ 대표 환자 후보 추출 완료


In [55]:
# ============================================
# 통계 요약
# ============================================

print("\n=== 감염 유형별 카운트 ===")
count_summary = representative_df.groupby(['infection_type', 'infection_timing']).agg(
    admission_count=('hadm_id', 'nunique'),
    patient_count=('subject_id', 'nunique')
).reset_index()
print(count_summary.to_string(index=False))

print("\n=== 감염 유형별 나이 분포 ===")
age_summary = representative_df.groupby('infection_type').agg(
    count=('hadm_id', 'nunique'),
    age_avg=('age', 'mean'),
    age_min=('age', 'min'),
    age_max=('age', 'max'),
    age_median=('age', 'median'),
    age_std=('age', 'std')
).round(1).reset_index()
print(age_summary.to_string(index=False))

print("\n=== 감염 유형별 재원일수 분포 ===")
los_summary = representative_df.groupby('infection_type').agg(
    count=('hadm_id', 'nunique'),
    los_avg=('los_days', 'mean'),
    los_min=('los_days', 'min'),
    los_max=('los_days', 'max'),
    los_median=('los_days', 'median'),
    los_std=('los_days', 'std')
).round(1).reset_index()
print(los_summary.to_string(index=False))

print("\n=== 성별 분포 ===")
gender_summary = representative_df.groupby(['infection_type', 'gender'])['hadm_id'].nunique().unstack(fill_value=0)
print(gender_summary)

print("\n=== 피벗 요약 (Admission vs Hospital_Acquired) ===")
pivot = count_summary.pivot(index='infection_type', columns='infection_timing', values='admission_count').fillna(0).astype(int)
pivot['Total'] = pivot.sum(axis=1)
print(pivot)


=== 감염 유형별 카운트 ===
infection_type  infection_timing  admission_count  patient_count
  GI_Infection         Admission                4              4
  GI_Infection Hospital_Acquired               58             57
     Pneumonia         Admission               16             16
     Pneumonia Hospital_Acquired               40             40
           UTI         Admission                1              1
           UTI Hospital_Acquired               52             52

=== 감염 유형별 나이 분포 ===
infection_type  count  age_avg  age_min  age_max  age_median  age_std
  GI_Infection     62     68.9       50       93        67.5     11.7
     Pneumonia     56     71.3       52       91        71.0     11.9
           UTI     53     70.7       50       93        71.0     10.7

=== 감염 유형별 재원일수 분포 ===
infection_type  count  los_avg  los_min  los_max  los_median  los_std
  GI_Infection     62     30.7      7.3     91.4        25.3     20.8
     Pneumonia     56     20.0      7.0    105.0        12.

---

## 3. 최종 코호트 추출 후 저장 (csv)

In [58]:
print("\n[코호트 통합] 전체 대표 환자 리스트 생성...")

# 1. MDRO 코호트 (최종 선택된 환자만)
print("\n--- MDRO 코호트 추출 ---")

# 최종 선택된 hadm_id 리스트
mrsa_final_hadm_ids = [22875063, 27165344, 25492887, 25292355, 28055582, 24195021, 25006173, 22793142]
cre_final_hadm_ids = [28029495, 22456860, 28540703, 25338284]

mdro_cohort_query = f"""
WITH mrsa_cases AS (
    SELECT hadm_id, 'MRSA' AS mdro_type
    FROM (
        SELECT 
            hadm_id,
            MAX(CASE WHEN ab_name IN ('OXACILLIN', 'CEFAZOLIN', 'CEFTRIAXONE', 'CEFEPIME') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS betalactam_r,
            MAX(CASE WHEN ab_name IN ('ERYTHROMYCIN', 'AZITHROMYCIN', 'CLARITHROMYCIN') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS macrolide_r,
            MAX(CASE WHEN ab_name IN ('LEVOFLOXACIN', 'CIPROFLOXACIN', 'MOXIFLOXACIN') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS quinolone_r
        FROM microbiologyevents
        WHERE org_name = 'STAPH AUREUS COAG +'
        GROUP BY hadm_id
    )
    WHERE betalactam_r = 1 AND macrolide_r = 1 AND quinolone_r = 1
    AND hadm_id IN ({','.join(map(str, mrsa_final_hadm_ids))})
),
cre_cases AS (
    SELECT hadm_id, 'CRE' AS mdro_type
    FROM (
        SELECT 
            hadm_id,
            MAX(CASE WHEN ab_name IN ('IMIPENEM', 'MEROPENEM', 'ERTAPENEM') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS carbapenem_r,
            MAX(CASE WHEN ab_name IN ('CEFTRIAXONE', 'CEFTAZIDIME', 'CEFEPIME') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS cephalosporin_r,
            MAX(CASE WHEN ab_name IN ('CIPROFLOXACIN', 'LEVOFLOXACIN', 'MOXIFLOXACIN') 
                     AND interpretation = 'R' THEN 1 ELSE 0 END) AS quinolone_r
        FROM microbiologyevents
        WHERE org_name LIKE '%KLEBSIELLA%' 
           OR org_name LIKE '%ESCHERICHIA%' 
           OR org_name LIKE '%ENTEROBACTER%'
        GROUP BY hadm_id
    )
    WHERE carbapenem_r = 1 AND cephalosporin_r = 1 AND quinolone_r = 1
    AND hadm_id IN ({','.join(map(str, cre_final_hadm_ids))})
),
all_mdro AS (
    SELECT hadm_id, mdro_type FROM mrsa_cases
    UNION
    SELECT hadm_id, mdro_type FROM cre_cases
)
SELECT 
    'MDRO' AS cohort_type,
    m.mdro_type AS subcategory,
    'N/A' AS infection_timing,
    a.subject_id,
    m.hadm_id,
    p.gender,
    (EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
     - CAST(p.anchor_year AS INTEGER) 
     + CAST(p.anchor_age AS INTEGER)) AS age,
    CAST(a.admittime AS DATE) AS admit_date,
    CAST(a.dischtime AS DATE) AS discharge_date,
    EXTRACT(EPOCH FROM (CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP))) / 86400 AS los_days
FROM all_mdro m
JOIN admissions a ON m.hadm_id = a.hadm_id
JOIN patients p ON a.subject_id = p.subject_id
"""

mdro_cohort = conn.execute(mdro_cohort_query).df()
print(f"  MDRO 코호트: {len(mdro_cohort):,}건")
print(f"    - MRSA: {len(mdro_cohort[mdro_cohort['subcategory'] == 'MRSA'])}건")
print(f"    - CRE: {len(mdro_cohort[mdro_cohort['subcategory'] == 'CRE'])}건")


[코호트 통합] 전체 대표 환자 리스트 생성...

--- MDRO 코호트 추출 ---
  MDRO 코호트: 12건
    - MRSA: 8건
    - CRE: 4건


In [59]:
# 2. 감염 코호트 (배양 4종 + 항생제 조건)
print("\n--- 감염 코호트 추출 (배양 4종 + 항생제 조건) ---")

infection_cohort_query = """
WITH culture_all_four AS (
    SELECT hadm_id
    FROM microbiologyevents
    WHERE spec_type_desc IN ('URINE', 'BLOOD CULTURE', 'STOOL', 'SPUTUM')
    GROUP BY hadm_id
    HAVING COUNT(DISTINCT spec_type_desc) = 4
),
abx_by_class AS (
    SELECT 
        hadm_id,
        MAX(CASE WHEN LOWER(medication) LIKE '%metronidazole%' THEN 1 ELSE 0 END) AS has_metronidazole,
        MAX(CASE WHEN LOWER(medication) LIKE '%floxacin%' THEN 1 ELSE 0 END) AS has_fluoroquinolone,
        MAX(CASE WHEN LOWER(medication) LIKE '%cef%' THEN 1 ELSE 0 END) AS has_cephalosporin,
        MAX(CASE WHEN LOWER(medication) LIKE '%azithromycin%' 
                 OR LOWER(medication) LIKE '%erythromycin%' 
                 OR LOWER(medication) LIKE '%clarithromycin%' THEN 1 ELSE 0 END) AS has_macrolide
    FROM pharmacy
    GROUP BY hadm_id
),
infection_diagnosis AS (
    SELECT 
        hadm_id,
        CASE 
            WHEN icd_code LIKE 'J12%' OR icd_code LIKE 'J13%' 
                 OR icd_code LIKE 'J14%' OR icd_code LIKE 'J15%'
                 OR icd_code LIKE 'J16%' OR icd_code LIKE 'J17%' 
                 OR icd_code LIKE 'J18%'
                 OR icd_code LIKE '480%' OR icd_code LIKE '481%'
                 OR icd_code LIKE '482%' OR icd_code LIKE '483%'
                 OR icd_code LIKE '484%' OR icd_code LIKE '485%'
                 OR icd_code LIKE '486%' OR icd_code LIKE '487%'
            THEN 'Pneumonia'
            WHEN icd_code LIKE 'A0%' OR icd_code LIKE 'A09%'
                 OR icd_code LIKE 'K52%'
                 OR icd_code LIKE '001%' OR icd_code LIKE '002%'
                 OR icd_code LIKE '003%' OR icd_code LIKE '004%'
                 OR icd_code LIKE '005%' OR icd_code LIKE '006%'
                 OR icd_code LIKE '007%' OR icd_code LIKE '008%'
                 OR icd_code LIKE '009%' OR icd_code LIKE '5589%'
            THEN 'GI_Infection'
            WHEN icd_code LIKE 'N390%' OR icd_code LIKE 'N391%'
                 OR icd_code LIKE 'N393%' OR icd_code LIKE 'N394%'
                 OR icd_code LIKE '5990%' OR icd_code LIKE '5997%'
            THEN 'UTI'
            ELSE NULL
        END AS infection_type,
        seq_num
    FROM diagnoses_icd
),
admission_infection AS (
    SELECT 
        hadm_id,
        infection_type,
        CASE WHEN MIN(seq_num) = 1 THEN 'Admission' ELSE 'Hospital_Acquired' END AS infection_timing
    FROM infection_diagnosis
    WHERE infection_type IS NOT NULL
    GROUP BY hadm_id, infection_type
),
filtered_admissions AS (
    SELECT 
        a.hadm_id,
        a.subject_id,
        a.admittime,
        a.dischtime,
        (EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
         - CAST(p.anchor_year AS INTEGER) 
         + CAST(p.anchor_age AS INTEGER)) AS age,
        p.gender,
        EXTRACT(EPOCH FROM (CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP))) / 86400 AS los_days
    FROM admissions a
    JOIN patients p ON a.subject_id = p.subject_id
    WHERE (
        EXTRACT(YEAR FROM CAST(a.admittime AS TIMESTAMP)) 
        - CAST(p.anchor_year AS INTEGER) 
        + CAST(p.anchor_age AS INTEGER)
    ) >= 50
    AND (
        CAST(a.dischtime AS TIMESTAMP) - CAST(a.admittime AS TIMESTAMP)
    ) >= INTERVAL '7 days'
    AND a.hadm_id NOT IN (SELECT DISTINCT hadm_id FROM icustays WHERE hadm_id IS NOT NULL)
    AND a.hadm_id IN (
        SELECT DISTINCT hadm_id 
        FROM transfers 
        WHERE careunit IN (
            'Medicine', 'Med/Surg', 'Medicine/Cardiology', 'Neurology',
            'Transplant', 'Hematology/Oncology', 'Vascular', 'Surgery/Trauma',
            'Med/Surg/GYN', 'Med/Surg/Trauma', 'Surgery', 'Cardiac Surgery',
            'Surgery/Pancreatic/Biliary/Bariatric', 'Surgery/Vascular/Intermediate',
            'Thoracic Surgery'
        )
    )
)
SELECT 
    'Infection' AS cohort_type,
    ai.infection_type AS subcategory,
    ai.infection_timing,
    fa.subject_id,
    ai.hadm_id,
    fa.gender,
    fa.age,
    CAST(fa.admittime AS DATE) AS admit_date,
    CAST(fa.dischtime AS DATE) AS discharge_date,
    fa.los_days
FROM admission_infection ai
JOIN filtered_admissions fa ON ai.hadm_id = fa.hadm_id
JOIN culture_all_four cf ON ai.hadm_id = cf.hadm_id
JOIN abx_by_class ab ON ai.hadm_id = ab.hadm_id
WHERE 
    (ai.infection_type = 'GI_Infection' AND ab.has_metronidazole = 1 AND ab.has_fluoroquinolone = 1)
    OR
    (ai.infection_type = 'Pneumonia' AND ab.has_cephalosporin = 1 AND ab.has_fluoroquinolone = 1 AND ab.has_macrolide = 1)
    OR
    (ai.infection_type = 'UTI' AND ab.has_fluoroquinolone = 1 AND ab.has_metronidazole = 1)
"""

infection_cohort = conn.execute(infection_cohort_query).df()
print(f"  감염 코호트: {len(infection_cohort):,}건")


--- 감염 코호트 추출 (배양 4종 + 항생제 조건) ---
  감염 코호트: 171건


In [60]:
# ============================================
# 통합 코호트 생성
# ============================================

print("\n--- 코호트 통합 ---")

# 컬럼 통일
mdro_cohort = mdro_cohort[['cohort_type', 'subcategory', 'infection_timing', 'subject_id', 'hadm_id', 
                           'gender', 'age', 'admit_date', 'discharge_date', 'los_days']]
infection_cohort = infection_cohort[['cohort_type', 'subcategory', 'infection_timing', 'subject_id', 'hadm_id', 
                                     'gender', 'age', 'admit_date', 'discharge_date', 'los_days']]

# 통합
full_cohort = pd.concat([mdro_cohort, infection_cohort], ignore_index=True)

# 중복 제거 (hadm_id 기준)
full_cohort = full_cohort.drop_duplicates(subset=['hadm_id', 'subcategory'])

print(f"\n✅ 전체 코호트: {len(full_cohort):,}건")
print(f"   Unique patients: {full_cohort['subject_id'].nunique():,}명")

# ============================================
# 요약 통계
# ============================================

print("\n=== 코호트별 요약 ===")
summary = full_cohort.groupby(['cohort_type', 'subcategory', 'infection_timing']).agg(
    count=('hadm_id', 'nunique'),
    patients=('subject_id', 'nunique'),
    age_avg=('age', 'mean'),
    age_min=('age', 'min'),
    age_max=('age', 'max'),
    los_avg=('los_days', 'mean'),
    los_min=('los_days', 'min'),
    los_max=('los_days', 'max')
).round(1).reset_index()

print(summary.to_string(index=False))


--- 코호트 통합 ---

✅ 전체 코호트: 183건
   Unique patients: 147명

=== 코호트별 요약 ===
cohort_type  subcategory  infection_timing  count  patients  age_avg  age_min  age_max  los_avg  los_min  los_max
  Infection GI_Infection         Admission      4         4     61.8       51       68     15.4      8.6     28.4
  Infection GI_Infection Hospital_Acquired     58        57     69.4       50       93     31.7      7.3     91.4
  Infection    Pneumonia         Admission     16        16     76.0       58       91     10.5      7.6     17.2
  Infection    Pneumonia Hospital_Acquired     40        40     69.4       52       91     23.8      7.0    105.0
  Infection          UTI         Admission      1         1     66.0       66       66     51.7     51.7     51.7
  Infection          UTI Hospital_Acquired     52        52     70.8       50       93     36.2      7.0    203.9
       MDRO          CRE               N/A      4         4     65.2       55       76     50.5     12.0    133.1
       MDRO   

In [61]:
# ============================================
# CSV 저장
# ============================================

# 전체 코호트 저장
full_cohort.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ 전체 코호트 저장: {OUTPUT_CSV}")

# 요약 통계 저장 (JSON)
summary.to_json(OUTPUT_JSON, orient='records', indent=2, force_ascii=False)
print(f"✅ 요약 통계 저장: {OUTPUT_JSON}")


✅ 전체 코호트 저장: ../outputs/cohort_patients_v4.csv
✅ 요약 통계 저장: ../outputs/cohort_patients_v4.json


In [62]:
# ============================================
# 최종 확인
# ============================================

print("\n=== 저장된 코호트 미리보기 ===")
print(full_cohort.head(20).to_string(index=False))

print("\n=== 코호트 구성 ===")
print(full_cohort.groupby(['cohort_type', 'subcategory'])['hadm_id'].nunique())


=== 저장된 코호트 미리보기 ===
cohort_type  subcategory  infection_timing subject_id  hadm_id gender  age admit_date discharge_date   los_days
       MDRO         MRSA               N/A   11438336 25292355      F   63 2132-05-11     2132-05-29  18.147917
       MDRO         MRSA               N/A   11644977 24195021      M   64 2135-04-28     2135-05-26  28.004861
       MDRO         MRSA               N/A   12356657 22793142      M   69 2155-08-16     2155-12-31 137.845833
       MDRO          CRE               N/A   12808803 28540703      M   76 2126-06-15     2126-07-30  44.847917
       MDRO         MRSA               N/A   16929830 25492887      F   56 2131-04-17     2131-05-01  13.796528
       MDRO          CRE               N/A   18294629 22456860      M   69 2143-09-23     2143-10-05  11.968056
       MDRO         MRSA               N/A   19432448 27165344      M   64 2119-02-09     2119-03-09  27.801389
       MDRO         MRSA               N/A   19440935 28055582      M   54 2126-11